# Segmentation + Depth vs Depth-only — **5-fold cross-validation** on NYU Depth V2

The k-fold version of **section 6** of `compare_depth_models_kaggle.ipynb`.

It evaluates the **segmentation+depth fusion model** — SAM2 segments every object, then each object's pixels are replaced by that object's **median depth** (object-median fusion), on top of a metric **Depth Anything V2** backbone — against the **same depth backbone without segmentation**. So any difference is attributable to the segmentation layer alone.

**What changed vs the 100-image section 6:**
- **3000 samples** instead of 100.
- Samples are split into **5 disjoint folds**; every metric is reported as a **mean ± std across the folds**, so the seg-vs-depth difference comes with a stability estimate rather than a single number.
- **Drawn from the `train` split** — NYU-V2 `validation` only has ~654 images, too few for 3000. This is still a fair test: the backbone (`Depth-Anything-V2-Metric-Indoor-*`) is trained on **Hypersim, not NYU**, so NYU images are unseen for it. *(Do not switch the baseline to an NYU-trained model like ZoeDepth — that would leak.)*

> **This is cross-validation for evaluation, not weight training.** Neither the depth model nor the object-median rule has trainable parameters; each image is scored once and the folds are aggregated.

**Before running:** enable **GPU** (Settings → Accelerator → GPU T4/P100) and **Internet** (Settings → Internet → On; needs a phone-verified Kaggle account).

**Heads-up on runtime:** SAM2 proposes masks for *every* object in *every* one of the 3000 images — this is slow (expect a few hours on a T4). The run is **resumable**: each image's scores are appended to `kfold_persample.jsonl` the moment it finishes, and re-running skips images already scored. If the session dies, just run the cell again.

> **Depth backbone for this notebook:** `Depth-Anything-V2-Metric-Indoor-Base` (`da2-metric-base`) — the *base* variant. Both the depth-only baseline and the SAM2-fused model use it, so the comparison stays apples-to-apples.

## 1. Install dependencies (only what's missing)

In [ ]:
import importlib.util, subprocess, sys
from importlib.metadata import version, PackageNotFoundError


def pkg_version(pkg):
    try:
        return tuple(int(p) for p in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


# Depth Pro is not used here, but transformers>=4.48 is a safe floor for the
# depth-estimation pipeline. Do NOT upgrade numpy/opencv/scipy on Kaggle --
# the prebuilt stack is a matched set and breaks if those change.
if pkg_version("transformers") < (4, 48):
    print("upgrading transformers...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.48"], check=True)
    print("transformers upgraded -- RESTART the session (Run > Restart & Clear), then re-run all cells.")

if pkg_version("datasets") == (0, 0):
    print("installing datasets...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

# SAM2 (installing a NEW package needs no restart, unlike upgrading one).
if importlib.util.find_spec("sam2") is None:
    print("installing SAM2 (a few minutes)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/facebookresearch/sam2.git"], check=True)

import transformers, datasets
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__, "| sam2 available")

## 2. Write the evaluation script

`seg_depth_kfold.py` is embedded below via `%%writefile`, so the notebook is **self-contained** — no Kaggle dataset attachment needed. It reuses the exact metrics, depth loader, SAM2 mask generator and object-median fusion from the repo's `eval_nyu_depth.py` / `sam2_depth_pipeline.py` / `eval_segmentation_depth.py`, and adds the resumable per-sample scoring and 5-fold aggregation.

In [ ]:
%%writefile seg_depth_kfold.py
#!/usr/bin/env python3
"""
seg_depth_kfold.py

5-FOLD CROSS-VALIDATION evaluation of the SEGMENTATION+DEPTH fusion model
(SAM2 masks + metric Depth Anything V2, object-median fusion) against the
SAME depth model WITHOUT segmentation, on NYU Depth V2.

This is the k-fold version of section 6 of compare_depth_models_kaggle.ipynb.
Instead of a single 100-image run it evaluates on --num-samples (default 3000)
images partitioned into 5 disjoint folds, and reports every metric as a
mean +/- std ACROSS the folds -- so the seg-vs-depth difference comes with a
measure of how stable it is, not a single point estimate.

IMPORTANT -- this is cross-validation for EVALUATION, not weight training.
Neither the depth model nor the object-median fusion rule has trainable
parameters here. Each image is scored exactly once (it belongs to one fold);
folds are then aggregated. "Fine-tuning with fivefold" here means the k-fold
evaluation protocol, giving a cross-validated mean+/-std comparison.

WHY THE TRAIN SPLIT
-------------------
The NYU-V2 validation split only has ~654 images -- not enough for 3000
samples, so we draw from the TRAIN split. This is still a fair test for the
depth backbone used here (Depth-Anything-V2-Metric-Indoor-*, trained on
Hypersim, NOT on NYU), so NYU train images are unseen data for it. Do NOT
reuse this protocol with an NYU-trained model (e.g. ZoeDepth-nyu) -- that
would be train/test leakage.

USAGE
-----
python seg_depth_kfold.py --num-samples 3000 --n-folds 5 --baseline da2-metric-small
python seg_depth_kfold.py --num-samples 200            # quick smoke test
python seg_depth_kfold.py --sam2-quick-test            # sparser/faster SAM2 grid

Resumable: per-sample scores are appended to <output-dir>/kfold_persample.jsonl
as each image finishes; re-running the same command skips images already
scored (essential -- 3000 images with SAM2 is long and Kaggle sessions die).
"""

import argparse
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import cv2

METRIC_FIELDS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "delta1", "delta2", "delta3"]
LOWER_BETTER = {"abs_rel", "sq_rel", "rmse", "rmse_log"}

# Depth-only baselines (metric checkpoints -> meters directly). Slugs match
# compare_depth_models.py so results stay comparable to the 7-model notebook.
BASELINE_IDS = {
    "da2-metric-small": "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
    "da2-metric-base":  "depth-anything/Depth-Anything-V2-Metric-Indoor-Base-hf",
    "da2-metric-large": "depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf",
}

# SAM2.1 model-size presets: checkpoint URL, config path, filename.
SAM2_PRESETS = {
    "tiny": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        config="configs/sam2.1/sam2.1_hiera_t.yaml", filename="sam2.1_hiera_tiny.pt"),
    "small": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt",
        config="configs/sam2.1/sam2.1_hiera_s.yaml", filename="sam2.1_hiera_small.pt"),
    "base_plus": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt",
        config="configs/sam2.1/sam2.1_hiera_b+.yaml", filename="sam2.1_hiera_base_plus.pt"),
    "large": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
        config="configs/sam2.1/sam2.1_hiera_l.yaml", filename="sam2.1_hiera_large.pt"),
}


# =========================================================================
# Standard monocular depth metrics (Eigen et al.; same as compare_depth_models)
# =========================================================================
def compute_depth_metrics(pred: np.ndarray, gt: np.ndarray,
                          min_depth: float = 1e-3, max_depth: float = 10.0) -> Optional[Dict[str, float]]:
    """AbsRel/SqRel/RMSE/RMSE-log (lower better) + delta1/2/3 (higher better).
    Returns None if the valid-pixel mask is empty."""
    valid = ((gt > min_depth) & (gt < max_depth) & np.isfinite(gt)
             & np.isfinite(pred) & (pred > min_depth))
    pv, gv = pred[valid], gt[valid]
    if pv.size == 0:
        return None
    ratio = np.maximum(pv / gv, gv / pv)
    return {
        "abs_rel": float(np.mean(np.abs(pv - gv) / gv)),
        "sq_rel": float(np.mean(((pv - gv) ** 2) / gv)),
        "rmse": float(np.sqrt(np.mean((pv - gv) ** 2))),
        "rmse_log": float(np.sqrt(np.mean((np.log(pv) - np.log(gv)) ** 2))),
        "delta1": float(np.mean(ratio < 1.25)),
        "delta2": float(np.mean(ratio < 1.25 ** 2)),
        "delta3": float(np.mean(ratio < 1.25 ** 3)),
    }


def clip_prediction(pred: np.ndarray, min_depth: float = 1e-3, max_depth: float = 10.0) -> np.ndarray:
    return np.clip(pred, min_depth, max_depth)

# =========================================================================
# NYU Depth V2 loading (streaming -- never holds all images in memory) and
# the depth model. Mirrors eval_nyu_depth.py (incl. the datasets>=4 parquet
# fallback) so behaviour matches the depth-only baseline exactly.
# =========================================================================
def iter_nyu(split: str, num_samples: int):
    """Yields (idx, hf_item) lazily. Conversion to numpy happens in the caller
    so already-scored indices can be skipped cheaply on a resumed run."""
    from datasets import load_dataset
    try:
        ds = load_dataset("sayakpaul/nyu_depth_v2", split=split, streaming=True)
    except RuntimeError:
        data_files = {split: f"hf://datasets/sayakpaul/nyu_depth_v2@refs/convert/parquet/default/partial-{split}/*.parquet"}
        ds = load_dataset("parquet", data_files=data_files, split=split, streaming=True)
    for i, item in enumerate(ds):
        if i >= num_samples:
            break
        yield i, item


def load_metric_depth_model(model_id: str, device: str):
    from transformers import pipeline
    return pipeline(task="depth-estimation", model=model_id, device=0 if device == "cuda" else -1)


def predict_depth_meters(depth_pipe, image_rgb: np.ndarray) -> np.ndarray:
    """Predicted depth in meters (metric checkpoint), resized to the image."""
    from PIL import Image
    result = depth_pipe(Image.fromarray(image_rgb))
    pred = result["predicted_depth"].squeeze().detach().cpu().numpy().astype(np.float32)
    if pred.shape != image_rgb.shape[:2]:
        pred = cv2.resize(pred, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    return pred


# =========================================================================
# SAM2 automatic mask generator (mirrors sam2_depth_pipeline.py)
# =========================================================================
def _download_with_progress(url: str, dest_path: str):
    import urllib.request
    def _hook(block_num, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = block_num * block_size
        pct = min(100, downloaded * 100 // total_size)
        print(f"\r  downloading {os.path.basename(dest_path)}: {pct}% "
              f"({downloaded/1e6:.0f}/{total_size/1e6:.0f} MB)", end="", flush=True)
    urllib.request.urlretrieve(url, dest_path, reporthook=_hook)
    print()


def ensure_sam2_checkpoint(model_size: str, checkpoints_dir: str) -> str:
    preset = SAM2_PRESETS[model_size]
    os.makedirs(checkpoints_dir, exist_ok=True)
    dest = os.path.join(checkpoints_dir, preset["filename"])
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"Using cached SAM2 checkpoint: {dest}")
        return dest
    print(f"Downloading SAM2 checkpoint ({model_size})...")
    _download_with_progress(preset["url"], dest)
    return dest


def load_sam2_mask_generator(sam2_checkpoint: str, sam2_config: str, device: str, quick_test: bool = False):
    from sam2.build_sam import build_sam2
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    sam2_model = build_sam2(sam2_config, sam2_checkpoint, device=device, apply_postprocessing=True)
    return SAM2AutomaticMaskGenerator(
        model=sam2_model,
        points_per_side=16 if quick_test else 32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        crop_n_layers=0 if quick_test else 1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )


def generate_masks(mask_generator, image_rgb: np.ndarray) -> List[Dict[str, Any]]:
    return sorted(mask_generator.generate(image_rgb), key=lambda m: m["area"], reverse=True)

# =========================================================================
# Object-median fusion (mirrors eval_segmentation_depth.py)
# =========================================================================
def _fit_mask(seg: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    if seg.shape[:2] != shape:
        seg = cv2.resize(seg.astype(np.uint8), (shape[1], shape[0]),
                         interpolation=cv2.INTER_NEAREST).astype(bool)
    return seg


def fuse_object_median(depth: np.ndarray, mask_dicts: List[Dict[str, Any]]) -> np.ndarray:
    """Replace every pixel inside a mask by the median of the RAW depth inside
    that mask. Largest masks first, so smaller (more specific) segments win on
    overlap; the median always comes from the original depth (no contamination)."""
    fused = depth.copy()
    for m in sorted(mask_dicts, key=lambda m: m["area"], reverse=True):
        seg = _fit_mask(m["segmentation"].astype(bool), depth.shape[:2])
        vals = depth[seg]
        if vals.size == 0:
            continue
        fused[seg] = np.median(vals)
    return fused


def union_masks(mask_dicts: List[Dict[str, Any]], shape: Tuple[int, int]) -> np.ndarray:
    union = np.zeros(shape, dtype=bool)
    for m in mask_dicts:
        union |= _fit_mask(m["segmentation"].astype(bool), shape)
    return union


def metrics_in_region(pred: np.ndarray, gt: np.ndarray, region: np.ndarray,
                      max_depth: float = 10.0) -> Optional[Dict[str, float]]:
    if not region.any():
        return None
    pred_r = np.where(region, pred, 0.0).astype(np.float32)
    gt_r = np.where(region, gt, 0.0).astype(np.float32)
    return compute_depth_metrics(pred_r, gt_r, max_depth=max_depth)


# =========================================================================
# Score one image: baseline (depth-only) vs fused (seg+depth), whole-image
# and inside/outside-mask, plus mask coverage.
# =========================================================================
def score_sample(base_pred: np.ndarray, gt: np.ndarray,
                 masks: List[Dict[str, Any]], max_depth: float) -> Optional[Dict[str, Any]]:
    fused = fuse_object_median(base_pred, masks)
    base_c = clip_prediction(base_pred, max_depth=max_depth)
    fused_c = clip_prediction(fused, max_depth=max_depth)
    b = compute_depth_metrics(base_c, gt, max_depth=max_depth)
    f = compute_depth_metrics(fused_c, gt, max_depth=max_depth)
    if b is None or f is None:
        return None
    union = union_masks(masks, gt.shape[:2])
    return {
        "base": b,
        "fused": f,
        "inside_base": metrics_in_region(base_c, gt, union, max_depth),
        "inside_fused": metrics_in_region(fused_c, gt, union, max_depth),
        "outside_base": metrics_in_region(base_c, gt, ~union, max_depth),
        "outside_fused": metrics_in_region(fused_c, gt, ~union, max_depth),
        "coverage": float(union.mean()),
        "fraction_changed": float((fused != base_pred).mean()),
        "n_masks": len(masks),
    }

# =========================================================================
# Resumable inference over all samples (the slow part -- run once)
# =========================================================================
def _free_gpu():
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def load_scored(jsonl_path: str) -> Dict[int, Dict[str, Any]]:
    """Loads already-scored per-sample rows keyed by sample index."""
    done: Dict[int, Dict[str, Any]] = {}
    if os.path.exists(jsonl_path):
        with open(jsonl_path) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    done[int(r["idx"])] = r
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


def run_inference(num_samples: int, split: str, baseline_slug: str, sam2_size: str,
                  output_dir: str, device: Optional[str], max_depth: float,
                  sam2_quick_test: bool, checkpoints_dir: str) -> str:
    """Scores baseline vs fused for every sample, appending to a JSONL. Skips
    indices already present, so re-running resumes where a dead session left off."""
    os.makedirs(output_dir, exist_ok=True)
    jsonl = os.path.join(output_dir, "kfold_persample.jsonl")
    done = load_scored(jsonl)
    print(f"{len(done)}/{num_samples} samples already scored (resuming).")
    if len(done) >= num_samples:
        print("Nothing to do -- all requested samples already scored.")
        return jsonl

    if device is None:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model_id = BASELINE_IDS[baseline_slug]
    print(f"device={device}  baseline={model_id}  sam2={sam2_size}"
          + ("  (quick grid)" if sam2_quick_test else ""))

    depth_pipe = load_metric_depth_model(model_id, device)
    sam2_ckpt = ensure_sam2_checkpoint(sam2_size, checkpoints_dir)
    sam2_cfg = SAM2_PRESETS[sam2_size]["config"]
    mask_gen = load_sam2_mask_generator(sam2_ckpt, sam2_cfg, device, quick_test=sam2_quick_test)

    n_new = n_skip = 0
    with open(jsonl, "a") as out:
        for idx, item in iter_nyu(split, num_samples):
            if idx in done:
                continue
            try:
                image = np.array(item["image"].convert("RGB"))
                gt = np.array(item["depth_map"]).astype(np.float32)
                base_pred = predict_depth_meters(depth_pipe, image)
                masks = generate_masks(mask_gen, image)
                row = score_sample(base_pred, gt, masks, max_depth)
            except Exception as e:
                n_skip += 1
                print(f"  [idx {idx}] skipped: {e}")
                continue
            if row is None:
                n_skip += 1
                continue
            row["idx"] = idx
            out.write(json.dumps(row) + "\n")
            out.flush()
            n_new += 1
            if n_new % 10 == 0 or (len(done) + n_new + n_skip) >= num_samples:
                print(f"  scored {len(done) + n_new}/{num_samples}  (skipped {n_skip})")
            if n_new % 50 == 0:
                _free_gpu()

    print(f"Inference done. {n_new} new samples scored, {n_skip} skipped.")
    _free_gpu()
    return jsonl

# =========================================================================
# 5-fold aggregation (pure post-processing over the cached scores)
# =========================================================================
def assign_folds(indices: List[int], n_folds: int, seed: int = 42) -> Dict[int, int]:
    """Deterministic disjoint folds: seeded shuffle of the sample indices,
    split into n_folds balanced chunks. Independent of the order samples were
    scored in, so resumed runs give the same folds."""
    idx = sorted(indices)
    perm = np.random.default_rng(seed).permutation(len(idx))
    fold_of: Dict[int, int] = {}
    for f, chunk in enumerate(np.array_split(perm, n_folds)):
        for pos in chunk:
            fold_of[idx[int(pos)]] = f
    return fold_of


def mean_metrics(rows: List[Dict[str, Any]], key: str) -> Dict[str, Optional[float]]:
    """Mean of each metric field over rows[*][key] (None entries skipped)."""
    acc: Dict[str, List[float]] = {f: [] for f in METRIC_FIELDS}
    for r in rows:
        m = r.get(key)
        if m is None:
            continue
        for f in METRIC_FIELDS:
            acc[f].append(m[f])
    return {f: (float(np.mean(acc[f])) if acc[f] else None) for f in METRIC_FIELDS}


def aggregate(jsonl_path: str, n_folds: int, seed: int, output_dir: str,
              baseline_slug: str) -> Dict[str, Any]:
    rows = list(load_scored(jsonl_path).values())
    if not rows:
        raise RuntimeError("No scored samples found -- run inference first.")
    fold_of = assign_folds([r["idx"] for r in rows], n_folds, seed)
    for r in rows:
        r["fold"] = fold_of[r["idx"]]

    # Per-fold means (each sample scored once, belongs to one fold).
    per_fold: Dict[int, Any] = {}
    for f in range(n_folds):
        fr = [r for r in rows if r["fold"] == f]
        per_fold[f] = {"n": len(fr), "base": mean_metrics(fr, "base"),
                       "fused": mean_metrics(fr, "fused")}

    # Across-fold mean/std (the headline CV numbers) + whole-dataset paired
    # improved/worsened/unchanged counts.
    summary: Dict[str, Any] = {}
    for field in METRIC_FIELDS:
        base_fold = [per_fold[f]["base"][field] for f in range(n_folds)
                     if per_fold[f]["base"][field] is not None]
        fused_fold = [per_fold[f]["fused"][field] for f in range(n_folds)
                      if per_fold[f]["fused"][field] is not None]
        diff_fold = [fu - b for b, fu in zip(base_fold, fused_fold)]
        d = [r["fused"][field] - r["base"][field] for r in rows]
        if field in LOWER_BETTER:
            improved = sum(x < -1e-9 for x in d)
            worsened = sum(x > 1e-9 for x in d)
        else:
            improved = sum(x > 1e-9 for x in d)
            worsened = sum(x < -1e-9 for x in d)
        summary[field] = {
            "base_mean": float(np.mean(base_fold)), "base_std": float(np.std(base_fold)),
            "fused_mean": float(np.mean(fused_fold)), "fused_std": float(np.std(fused_fold)),
            "diff_mean": float(np.mean(diff_fold)), "diff_std": float(np.std(diff_fold)),
            "improved": improved, "worsened": worsened, "unchanged": len(d) - improved - worsened,
        }

    region = {reg: {m: mean_metrics(rows, f"{reg}_{m}") for m in ("base", "fused")}
              for reg in ("inside", "outside")}
    pixel = {"mean_coverage": float(np.mean([r["coverage"] for r in rows])),
             "mean_fraction_changed": float(np.mean([r["fraction_changed"] for r in rows]))}

    result = {"n_samples": len(rows), "n_folds": n_folds, "seed": seed,
              "baseline_slug": baseline_slug, "fusion": "object-median",
              "per_fold": per_fold, "summary": summary, "region": region, "pixel_stats": pixel}
    with open(os.path.join(output_dir, "kfold_summary.json"), "w") as fh:
        json.dump(result, fh, indent=2)
    return result

# =========================================================================
# CSV + plots
# =========================================================================
def write_csvs(result: Dict[str, Any], output_dir: str):
    p1 = os.path.join(output_dir, "kfold_per_fold.csv")
    with open(p1, "w", newline="") as fh:
        fh.write("fold,n,model," + ",".join(METRIC_FIELDS) + "\n")
        for f in range(result["n_folds"]):
            for model in ("base", "fused"):
                m = result["per_fold"][f][model]
                vals = ",".join("" if m[k] is None else f"{m[k]:.6f}" for k in METRIC_FIELDS)
                fh.write(f"{f},{result['per_fold'][f]['n']},{model},{vals}\n")

    p2 = os.path.join(output_dir, "kfold_summary.csv")
    with open(p2, "w", newline="") as fh:
        fh.write("metric,base_mean,base_std,fused_mean,fused_std,"
                 "diff_mean,diff_std,improved,worsened,unchanged\n")
        for field in METRIC_FIELDS:
            s = result["summary"][field]
            fh.write(f"{field},{s['base_mean']:.6f},{s['base_std']:.6f},"
                     f"{s['fused_mean']:.6f},{s['fused_std']:.6f},"
                     f"{s['diff_mean']:.6f},{s['diff_std']:.6f},"
                     f"{s['improved']},{s['worsened']},{s['unchanged']}\n")
    print(f"Saved {p1}\nSaved {p2}")


def plot_summary(result: Dict[str, Any], output_dir: str, baseline_name: str, fused_name: str):
    import matplotlib.pyplot as plt
    c_b, c_f = plt.get_cmap("tab10")(0), plt.get_cmap("tab10")(1)
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.ravel()
    for ax, field in zip(axes, METRIC_FIELDS):
        s = result["summary"][field]
        bars = ax.bar([0, 1], [s["base_mean"], s["fused_mean"]],
                      yerr=[s["base_std"], s["fused_std"]], capsize=6, color=[c_b, c_f])
        for x, v in zip([0, 1], [s["base_mean"], s["fused_mean"]]):
            ax.text(x, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        lower = field in LOWER_BETTER
        ax.set_title(f"{field}  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([baseline_name, fused_name], fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    axes[7].axis("off")
    ar, d1 = result["summary"]["abs_rel"], result["summary"]["delta1"]
    axes[7].text(
        0.02, 0.95,
        f"{result['n_folds']}-fold CV over {result['n_samples']} images\n"
        f"error bars = std across folds\n\n"
        f"AbsRel: {ar['base_mean']:.4f} -> {ar['fused_mean']:.4f}\n"
        f"  diff {ar['diff_mean']:+.4f} +/- {ar['diff_std']:.4f}\n"
        f"  improved {ar['improved']}, worsened {ar['worsened']}, unchanged {ar['unchanged']}\n\n"
        f"delta1: {d1['base_mean']:.4f} -> {d1['fused_mean']:.4f}\n"
        f"  diff {d1['diff_mean']:+.4f} +/- {d1['diff_std']:.4f}\n\n"
        f"mask coverage {result['pixel_stats']['mean_coverage']*100:.1f}% of pixels\n"
        f"pixels changed by fusion {result['pixel_stats']['mean_fraction_changed']*100:.1f}%",
        transform=axes[7].transAxes, va="top", fontsize=10)

    fig.suptitle(f"Depth-only vs SAM2+depth (object-median fusion) -- "
                 f"{result['n_folds']}-fold CV, same backbone", fontsize=15)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_comparison.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_per_fold(result: Dict[str, Any], output_dir: str):
    import matplotlib.pyplot as plt
    nf = result["n_folds"]
    x = np.arange(nf)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, field in zip(axes, ["abs_rel", "delta1"]):
        base = [result["per_fold"][f]["base"][field] for f in range(nf)]
        fused = [result["per_fold"][f]["fused"][field] for f in range(nf)]
        ax.plot(x, base, "o-", label="depth-only")
        ax.plot(x, fused, "s-", label="seg+depth (fused)")
        lower = field in LOWER_BETTER
        ax.set_title(f"{field} per fold  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels([f"fold {i}" for i in range(nf)])
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Per-fold stability of the seg-vs-depth comparison", fontsize=14)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_per_fold.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_region(result: Dict[str, Any], output_dir: str, baseline_name: str, fused_name: str):
    import matplotlib.pyplot as plt
    reg = result["region"]
    c_b, c_f = plt.get_cmap("tab10")(0), plt.get_cmap("tab10")(1)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, field in zip(axes, ["abs_rel", "rmse", "delta1"]):
        vb = [reg["inside"]["base"][field], reg["outside"]["base"][field]]
        vf = [reg["inside"]["fused"][field], reg["outside"]["fused"][field]]
        vb = [0.0 if v is None else v for v in vb]
        vf = [0.0 if v is None else v for v in vf]
        x = np.arange(2)
        ax.bar(x - 0.18, vb, width=0.36, label=baseline_name, color=c_b)
        ax.bar(x + 0.18, vf, width=0.36, label=fused_name, color=c_f)
        lower = field in LOWER_BETTER
        ax.set_title(f"{field}  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(["inside masks", "outside masks"], fontsize=9)
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Where fusion acts: error inside vs outside SAM2 masks "
                 "(outside bars should nearly match -- sanity check)", fontsize=13)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_region.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")

# =========================================================================
# Orchestration
# =========================================================================
def print_verdict(result: Dict[str, Any], baseline_name: str):
    ar = result["summary"]["abs_rel"]
    d1 = result["summary"]["delta1"]
    print(f"\n--- {result['n_folds']}-fold verdict over {result['n_samples']} images "
          f"(vs {baseline_name}) ---")
    print(f"AbsRel: {ar['base_mean']:.4f}+/-{ar['base_std']:.4f} -> "
          f"{ar['fused_mean']:.4f}+/-{ar['fused_std']:.4f}  "
          f"(diff {ar['diff_mean']:+.4f}+/-{ar['diff_std']:.4f}; "
          f"improved {ar['improved']}, worsened {ar['worsened']}, unchanged {ar['unchanged']})")
    print(f"delta1: {d1['base_mean']:.4f}+/-{d1['base_std']:.4f} -> "
          f"{d1['fused_mean']:.4f}+/-{d1['fused_std']:.4f}  "
          f"(diff {d1['diff_mean']:+.4f}+/-{d1['diff_std']:.4f})")


def run_kfold(num_samples: int = 3000, n_folds: int = 5, split: str = "train",
              baseline_slug: str = "da2-metric-small", sam2_size: str = "small",
              output_dir: str = "seg_depth_kfold_out", device: Optional[str] = None,
              max_depth: float = 10.0, seed: int = 42, sam2_quick_test: bool = False,
              checkpoints_dir: str = "checkpoints") -> Dict[str, Any]:
    """End-to-end: resumable inference -> 5-fold aggregation -> CSVs + plots.
    Safe to call repeatedly; inference resumes and aggregation is recomputed."""
    jsonl = run_inference(num_samples, split, baseline_slug, sam2_size, output_dir,
                          device, max_depth, sam2_quick_test, checkpoints_dir)
    result = aggregate(jsonl, n_folds, seed, output_dir, baseline_slug)
    write_csvs(result, output_dir)
    fused_name = f"SAM2 + {baseline_slug}"
    plot_summary(result, output_dir, baseline_slug, fused_name)
    plot_per_fold(result, output_dir)
    plot_region(result, output_dir, baseline_slug, fused_name)
    print_verdict(result, baseline_slug)
    return result


def main():
    ap = argparse.ArgumentParser(
        description="5-fold CV of SAM2+depth (object-median) vs depth-only on NYU Depth V2")
    ap.add_argument("--split", default="train", help="'train' (default; validation too small for 3000) or 'validation'")
    ap.add_argument("--num-samples", type=int, default=3000)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--baseline", default="da2-metric-small", choices=list(BASELINE_IDS))
    ap.add_argument("--sam2-size", default="small", choices=list(SAM2_PRESETS))
    ap.add_argument("--output-dir", default="seg_depth_kfold_out")
    ap.add_argument("--checkpoints-dir", default="checkpoints")
    ap.add_argument("--device", default=None, help="cuda | cpu (auto-detected if omitted)")
    ap.add_argument("--max-depth", type=float, default=10.0)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--sam2-quick-test", action="store_true", help="Sparser SAM2 grid (faster, coarser)")
    a = ap.parse_args()
    run_kfold(num_samples=a.num_samples, n_folds=a.n_folds, split=a.split,
              baseline_slug=a.baseline, sam2_size=a.sam2_size, output_dir=a.output_dir,
              device=a.device, max_depth=a.max_depth, seed=a.seed,
              sam2_quick_test=a.sam2_quick_test, checkpoints_dir=a.checkpoints_dir)


if __name__ == "__main__":
    main()

In [ ]:
import importlib, inspect
import seg_depth_kfold as K
importlib.reload(K)  # pick up whatever the %%writefile cell just wrote to disk

assert "iter_nyu" in dir(K) and "run_kfold" in dir(K), "script did not load correctly"
assert "partial-" in inspect.getsource(K.iter_nyu), "stale script (no datasets>=4 parquet fallback)"
print("loaded seg_depth_kfold |", len(K.BASELINE_IDS), "baselines,", len(K.SAM2_PRESETS), "SAM2 sizes")

## 3. Configuration

In [ ]:
# -----------------------------------------------------------------------
# CONFIGURATION -- edit these
# -----------------------------------------------------------------------
NUM_SAMPLES = 3000          # total NYU images, split across the folds
N_FOLDS     = 5             # 5-fold cross-validation
SPLIT       = "train"       # 'validation' has only ~654 imgs -> use 'train' for 3000

# Depth backbone shared by BOTH the depth-only baseline and the fused model.
# One of: "da2-metric-small" | "da2-metric-base" | "da2-metric-large"
BASELINE_SLUG = "da2-metric-base"

# SAM2 size: "tiny" | "small" | "base_plus" | "large" (bigger = slower, finer masks)
SAM2_SIZE   = "small"

SEED        = 42            # fold assignment seed (reproducible folds)
OUTPUT_DIR  = "/kaggle/working/seg_depth_kfold_out"

# Sparser SAM2 point grid -> faster but coarser. Leave False for the real run;
# flip to True (and drop NUM_SAMPLES to ~50) for a quick end-to-end smoke test.
SAM2_QUICK_TEST = False

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE} | {N_FOLDS}-fold over {NUM_SAMPLES} '{SPLIT}' images | "
      f"baseline={BASELINE_SLUG} | sam2={SAM2_SIZE}")

## 4. Run the 5-fold comparison

This runs the slow inference **once** over all `NUM_SAMPLES` images (baseline depth + SAM2 masks + fused, scored per image), appending each result to `kfold_persample.jsonl`, then aggregates into 5 folds and writes the CSVs + charts.

**Resumable:** if the Kaggle session dies partway, just re-run this cell — already-scored images are skipped and it continues. You can also run it with a small `NUM_SAMPLES` first to sanity-check, then raise it; the earlier scores are reused.

In [ ]:
result = K.run_kfold(
    num_samples=NUM_SAMPLES,
    n_folds=N_FOLDS,
    split=SPLIT,
    baseline_slug=BASELINE_SLUG,
    sam2_size=SAM2_SIZE,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_depth=10.0,
    seed=SEED,
    sam2_quick_test=SAM2_QUICK_TEST,
)
print("\nDone. Outputs in:", OUTPUT_DIR)

## 5. Results

In [ ]:
import os, json
import pandas as pd
from IPython.display import Image, display

with open(f"{OUTPUT_DIR}/kfold_summary.json") as f:
    res = json.load(f)

print(f"{res['n_folds']}-fold CV over {res['n_samples']} images | "
      f"baseline={res['baseline_slug']} | fusion={res['fusion']}")
print(f"mean mask coverage: {res['pixel_stats']['mean_coverage']*100:.1f}% of pixels | "
      f"pixels changed by fusion: {res['pixel_stats']['mean_fraction_changed']*100:.1f}%\n")

print("Cross-validated summary (mean +/- std across folds):")
display(pd.read_csv(f"{OUTPUT_DIR}/kfold_summary.csv"))

print("\nPer-fold means:")
display(pd.read_csv(f"{OUTPUT_DIR}/kfold_per_fold.csv"))

for fig in ["kfold_comparison.png", "kfold_per_fold.png", "kfold_region.png"]:
    p = f"{OUTPUT_DIR}/{fig}"
    if os.path.exists(p):
        display(Image(p))

## 6. Download all outputs at once (zip)

Run this **after** the run in section 4 finishes. It bundles the entire `OUTPUT_DIR` — the per-sample `kfold_persample.jsonl`, `kfold_summary.json`, both CSVs and every PNG chart — into a **single zip** under `/kaggle/working`, so you can grab everything in one click from the right-side **Output** panel instead of downloading file by file.

In [ ]:
# -----------------------------------------------------------------------
# Bundle the WHOLE output directory into one downloadable zip.
# Lives under /kaggle/working so it shows in the right-side Output panel.
# -----------------------------------------------------------------------
import os, shutil
from datetime import datetime
from IPython.display import FileLink, display

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = f"/kaggle/working/seg_depth_kfold_{BASELINE_SLUG}_{stamp}"

# make_archive zips the entire tree rooted at OUTPUT_DIR into <zip_base>.zip
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_DIR)
size_mb = os.path.getsize(zip_path) / 1e6

print(f"Bundled {OUTPUT_DIR}\n     -> {zip_path}  ({size_mb:.1f} MB)\n")
print("Contents:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, name)
    if os.path.isfile(fp):
        print(f"  {name:34s} {os.path.getsize(fp)/1e6:7.2f} MB")

# Clickable download link (Kaggle/Jupyter). If the cwd isn't /kaggle/working
# the link may not resolve -- the zip is still in the Output panel regardless.
try:
    display(FileLink(os.path.relpath(zip_path, os.getcwd())))
except Exception as e:
    print(f"\n(download link unavailable: {e} -- use the right-side Output panel)")


## Notes for interpreting the results

- **Error bars in the summary chart are the std across the 5 folds** — small bars mean the seg-vs-depth difference is stable, not an artefact of one lucky split. The `diff_std` column in `kfold_summary.csv` is the same thing per metric.
- **`improved` / `worsened` / `unchanged`** count *images* (out of `NUM_SAMPLES`), from the paired per-image difference (fused − baseline). This is the paired verdict: on how many images did adding segmentation actually help.
- **Inside vs outside masks** (`kfold_region.png`): fusion only rewrites pixels *inside* SAM2 masks, so the *outside* bars for baseline and fused should be near-identical — a sanity check. If the whole-image means barely move but the *inside* bars differ, the effect is real but diluted by unmasked background.
- **Object-median fusion enforces piecewise-constant depth per object.** It removes within-object noise but erases genuine depth variation on slanted/curved surfaces (a floor, a wall seen at an angle) — so expect it to help on flat fronto-parallel objects and hurt on large slanted ones.
- **Reproducibility:** folds are fixed by `SEED`; the same seed + sample set always gives the same folds.
- **Download outputs** from the right-side Output panel before the session ends — Kaggle wipes `/kaggle/working`. Keep `kfold_persample.jsonl` if you want to resume or re-fold later without re-running inference.